# RunPod · Qwen3.5-122B-A10B bf16 LoRA → 검증 → test → 결과 올리기

stage2와 **학습·평가·제출 방식이 완전히 같습니다**(분할 md5, 프롬프트, 정답 글자 손실, 선지 셔플, LoRA r16, 1 epoch, up2.0 평가).
결과 파일 이름도 같아서, Drive `vqa_runs/q35_122b_a10b_r16_native_bf16/`에 넣으면 ens_search·manual_ensemble이 새 멤버로 바로 읽습니다.

**실행 순서** (자세한 RunPod 사용법은 가이드 문서 참고)
1. `DRY_RUN = True` 그대로 두고 메뉴 **Run → Run All Cells**
   - 처음 한 번은 설치 약 5분, 데이터 받기 몇 분, 가중치 244GB 받기가 걸립니다.
   - 3스텝만 학습한 뒤, 마지막에 **본 학습 예상 시간·비용 표**가 나옵니다.
2. 이상이 없으면 1번 셀에서 `DRY_RUN = False`로 바꾸고 **Kernel → Restart Kernel and Run All Cells**
   - 설치·데이터·가중치는 이미 있어서 바로 넘어갑니다.
   - zero-shot 300문항 → 학습(약 357스텝) → 검증 1,000 → test 6,714 순서로 진행합니다.
3. 끝나면 마지막 14번 셀이 결과를 HF 비공개 저장소로 올립니다 → Colab 브리지 노트북 **B 단계**로 Drive에 받기 → RunPod pod **Terminate**

**중간에 끊겨도 됩니다**
- 브라우저를 닫아도 학습은 계속됩니다. 진행 상황은 `/workspace/vqa_runs/<실행 이름>/progress.log`에 계속 적힙니다.
- pod가 멈췄다 다시 켜졌다면 1번 셀부터 다시 실행하세요. 볼륨 디스크의 체크포인트에서 이어서 학습합니다(25스텝마다 저장).


## 0. RunPod 환경 점검 (가장 먼저)
- GPU 메모리 합계, 드라이버, 디스크 공간을 확인합니다.
  - GPU 메모리 합계 250GiB 이상: B300 1장 또는 B200 2장
  - 드라이버 575 이상: CUDA 12.9 이상을 쓰기 위해
  - `/workspace` 남은 공간 300GB 이상: 가중치 244GB + 데이터·결과
  - 컨테이너 디스크 남은 공간 15GB 이상: 라이브러리 설치용
- 여기서 torch를 불러오지 않으므로, 2번 셀이 torch를 새로 설치해도 커널을 다시 시작할 필요가 없습니다.


In [ ]:
import os, sys, shutil, subprocess, re
os.environ.setdefault("HF_HOME", "/workspace/hf")               # 가중치 캐시는 반드시 볼륨 디스크(/workspace)에 — 컨테이너 디스크는 작고 pod를 멈추면 지워짐
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
WORK = "/workspace"
assert os.path.isdir(WORK), "/workspace가 없습니다 → pod 설정에서 Volume Disk(마운트 경로 /workspace)를 켰는지 확인하세요"

_q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap", "--format=csv,noheader,nounits"], capture_output=True, text=True)
assert _q.returncode == 0 and _q.stdout.strip(), "nvidia-smi 실패 → GPU pod인지 확인하세요\n" + (_q.stderr or "")
GPUS = []
for _line in _q.stdout.strip().splitlines():
    _name, _mem, _drv, _cap = [x.strip() for x in _line.split(",")[:4]]
    GPUS.append({"name": _name, "mem_gib": float(_mem) / 1024, "driver": _drv, "cap": _cap})
for _i, _g in enumerate(GPUS): print(f"GPU{_i}: {_g['name']} | {_g['mem_gib']:.0f}GiB | 드라이버 {_g['driver']} | compute {_g['cap']}")
TOTAL_VRAM_GIB = sum(_g["mem_gib"] for _g in GPUS)
DRIVER_MAJOR = int(GPUS[0]["driver"].split(".")[0])
_dw, _dc = shutil.disk_usage(WORK), shutil.disk_usage("/")
_ram = int(re.search(r"MemTotal:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
print(f"볼륨 디스크 /workspace: 남은 {_dw.free / 1e9:.0f}GB / 전체 {_dw.total / 1e9:.0f}GB | 컨테이너 디스크: 남은 {_dc.free / 1e9:.0f}GB | RAM {_ram:.0f}GB")
_have_w = os.path.isdir(os.path.join(os.environ["HF_HOME"], "hub", "models--Qwen--Qwen3.5-122B-A10B", "snapshots"))
_prob = []
if TOTAL_VRAM_GIB < 250: _prob.append(f"GPU 메모리 합계 {TOTAL_VRAM_GIB:.0f}GiB < 250GiB → 122B bf16(약 228GiB)이 안 들어갑니다. B300 1장 또는 B200 2장으로 pod를 다시 만드세요")
if DRIVER_MAJOR < 575: _prob.append(f"드라이버 {GPUS[0]['driver']} < 575 → CUDA 12.9 이상을 못 씁니다. 다른 pod(다른 서버)를 새로 만드세요")
if _dw.free < 300e9 and not _have_w: _prob.append(f"/workspace 남은 공간 {_dw.free / 1e9:.0f}GB < 300GB → pod의 Volume Disk를 600GB로 늘리세요 (Pods → ⋮ → Edit Pod)")
if _dc.free < 15e9: _prob.append(f"컨테이너 디스크 남은 공간 {_dc.free / 1e9:.0f}GB < 15GB → Container Disk를 50GB로 늘리세요 (Edit Pod)")
if _prob: raise SystemExit("❌ 환경 문제:\n  - " + "\n  - ".join(_prob))
if len(GPUS) > 1: print(f"GPU {len(GPUS)}장 → 모델을 층 단위로 나눠 올립니다 (속도는 한 장과 비슷)")
if TOTAL_VRAM_GIB < 270: print("⚠️ GPU 메모리 합계가 빠듯합니다 (270GiB 미만). 학습 중 OOM이 나면 자동으로 마이크로 배치를 줄입니다")
print("✅ 환경 OK — 1번 셀로 넘어가세요" + (" (가중치는 이미 받아 둠)" if _have_w else ""))


## 1. 실험 설정


In [ ]:
RUN_NAME   = "q35_122b_a10b_r16_native_bf16"   # 저장 폴더 이름 = ens_search가 읽는 멤버 이름. 설정을 바꾸면 이름도 바꾸세요
MODEL_ID   = "Qwen/Qwen3.5-122B-A10B"          # bf16 약 244GB → B300(288GB) 1장 또는 B200 2장
DRY_RUN    = True           # ① True: 3스텝만 학습해 메모리·속도·비용 확인 → ② 통과하면 False로 바꾸고 Kernel → Restart Kernel and Run All Cells
ZERO_SHOT_N      = 300      # 학습 전에 기본 모델로 검증 앞 N문항 채점 (한국어 읽기 실력·답 위치 점검). 0이면 생략
ZERO_SHOT_ONLY   = False    # True면 zero-shot만 하고 멈춤 (122B는 GPTQ 4bit로 이미 96.0% 확인 → 바로 학습)
ZERO_SHOT_PRESET = "up2.0"  # zero-shot 채점 해상도 (기존 기록 Q35 제로샷 96.1%가 up2.0)
ZERO_SHOT_MIN_ACC = 0.93    # 본 학습에서 zero-shot이 이보다 낮으면 학습 전에 멈춤 (프롬프트·해상도·커널 고장을 비싼 학습 전에 잡음). 참고: GPTQ 4bit zero-shot 96.0%
GPU_PRICE_PER_HOUR = 7.89   # DRY_RUN 끝 예상 비용 계산용 시간당 요금 ($). B300 1장 Secure 7.89 · Community 6.94 · B200 2장 13.58
USE_FLA    = True           # 선형 어텐션 고속 커널(flash-linear-attention). DRY_RUN에서 fla/Triton 오류가 나면 False → Kernel → Restart Kernel and Run All Cells (느려지지만 동작)

# ── 학습 ──
TRAIN_PRESET = "native"    # 학습 해상도
EPOCHS     = 1
MAX_STEPS  = None          # 숫자를 넣으면 그 스텝에서 학습을 끝냄 (큰 모델을 짧게 학습할 때. 예: 200). None이면 EPOCHS만큼
LR         = 1e-4
MICRO_BS   = 2             # GPU에 한 번에 올리는 문항 수. 학습 중 OOM이 나면 자동으로 절반으로 줄이고 GRAD_ACCUM을 2배로 늘립니다
GRAD_ACCUM = 8             # 유효 배치 = MICRO_BS x GRAD_ACCUM = 16 (이 곱이 같으면 학습 결과의 성격은 같습니다)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
WARMUP_RATIO = 0.05
SHUFFLE_OPTIONS = True     # 선지 셔플 증강
TRAIN_ON_FULL = False      # True: 검증셋까지 합쳐 전체 6,714개로 학습 (레시피가 확정된 뒤 최종 제출용)
LOG_EVERY, EVAL_EVERY, SAVE_EVERY = 10, 100, 25    # 옵티마이저 스텝 단위 (체크포인트는 LoRA뿐이라 작고 빠름 → 25스텝마다)
QUICK_EVAL_N = 300         # 학습 도중 빠른 점검용 검증 문항 수 (epoch 끝에는 1,000개 전체 평가)

# ── 평가/제출 ──
EVAL_PRESETS = ["up2.0"]   # 학습이 끝난 뒤 이 해상도들로 검증셋 전체 평가
EVAL_TTA     = False        # 가장 좋았던 해상도에서 선지 순열 4종 평균(TTA)의 효과도 측정
MAKE_SUBMISSION = True      # 학습·평가가 끝나면 이어서 test 추론 → 제출 파일까지 생성 (ens_search에 test 예측이 필요)
TEST_PRESET  = "up2.0"      # "auto": 검증 정확도가 가장 높았던 해상도 / 또는 "native", "up1.5" 직접 지정
TEST_TTA     = False       # True면 test 추론 시간이 4배. 검증에서 TTA 이득이 확인된 뒤 켜서 12번 셀만 다시 실행

# ── RunPod 경로 (바꾸지 마세요. 모두 볼륨 디스크 /workspace 안) ──
ZIP_PATH     = "/workspace/ssafy-16-2-ai.zip"   # 대회 데이터 zip. 없으면 3번 셀이 HF 비공개 저장소에서 받음 (Jupyter로 직접 올려도 됨)
ZIP_PASSWORD = "YOUR_ZIP_PASSWORD"
DATA_DIR     = "/workspace/data"
OUT_DIR      = "/workspace/vqa_runs"
HF_REPO      = ""            # 비워 두면 "<내 HF 아이디>/ssafy-vqa-private" (Colab 브리지 노트북과 같은 규칙)
HF_ZIP_NAME  = "ssafy-16-2-ai.zip"
USE_4BIT   = False         # bf16 LoRA (122B는 MoE 전문가 층 때문에 4bit 학습이 안 됨)
BATCH_SIZE = "auto"        # 평가용 배치
SEED, VAL_SIZE = 42, 1000  # 검증 분할용 — 팀 공통값이므로 절대 바꾸지 마세요
TRAIN_SEED = 42            # 학습 순서·선지 셔플·LoRA 초기화 시드. 같은 레시피로 두 번째 모델을 만들 때(시드 앙상블) 43 등으로 변경
TRAIN_LIMIT = None         # 디버그용: 학습 샘플 수 제한
EXTRA_TRAIN_CSV = None     # 추가 학습 데이터 CSV 경로 (예: Stage 5가 만든 dev 의사 라벨). 열: id,path,question,a,b,c,d,answer. 검증 문항은 자동 차단
IMG_AUG = False            # True: 학습 이미지에만 밝기·대비·채도·감마·JPEG 화질을 무작위로 흔듦 (글자·방향·위치는 그대로). 평가·test에는 적용 안 함
TRAIN_PRESETS = None       # 예: ["native", "up1.5", "up2.0"] → 마이크로 배치마다 학습 해상도를 무작위로 고름(다해상도 학습). None이면 TRAIN_PRESET 고정
SKIP_MISSING_IMAGES = False
# ── (다른 계열용 설정 — 122B에서는 쓰이지 않음) ──
MUSE_REASONING = "low"     # Muse Glimmer 전용: 시스템 프롬프트의 추론 강도 (기본 high). 답 한 글자만 보므로 low
TEMPLATE_DATE  = "2026-09-27"   # 템플릿이 '오늘 날짜'를 프롬프트에 넣는 모델(Muse)은 날짜를 고정 → 학습·검증·test 입력이 날마다 달라지지 않음(재현성)
MUSE_ANSWER_PREFIX = "정답: ("  # Muse 전용: 답 채널(<|message|>) 뒤에 붙여 다음 토큰이 곧 답 글자가 되게 함 (진단: letter_mass 0.000 → 0.999, 16문항 16개 정답)
GEN_SUFFIX_OVERRIDE = None     # 답 글자 바로 앞에 붙일 문자열을 직접 지정 (None: 자동). 학습·검증·test 모두 같은 값이 쓰임
HF_TOKEN   = ""            # 비워 두면 pod 환경 변수 HF_TOKEN → 그것도 없으면 3번 셀에서 입력창이 뜸 (Write 권한 토큰)

RES_PRESETS = {
    "baseline384": (384 * 384, 384 * 384),
    "mid":         (256 * 28 * 28, 512 * 28 * 28),
    "native":      (256 * 28 * 28, 1280 * 28 * 28),
    "up1.5":       (1_550_000, 2_100_000),
    "up2.0":       (2_700_000, 3_400_000),
}
# Gemma 4 계열은 이미지당 '소프트 토큰' 수(70/140/280/560/1120)로 해상도를 정합니다. 프리셋 이름은 그대로 쓰고 값만 대응 (280 ≈ 원본, 560 ≈ 1.3배, 1120 ≈ 1.9배)
RES_PRESETS_GEMMA4 = {"baseline384": 70, "mid": 140, "native": 280, "up1.5": 560, "up2.0": 1120}
RES_PRESET = TRAIN_PRESET
import os; os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # GPU 메모리 단편화 완화 (torch import 전에 설정)


## 2. 라이브러리 설치 (RunPod용)
- **torch**: B300(Blackwell Ultra)은 CUDA 12.9 이상으로 빌드된 torch가 필요합니다.
  - 지금 torch가 그보다 낮거나 GPU 계산이 안 되면 CUDA 13.0용 휠로 바꿉니다(드라이버 580 미만이면 12.9용).
  - 이 경우 2~5분 걸립니다.
- 오래된 torchaudio는 지웁니다. 새 torch와 CUDA 버전이 달라 transformers의 프로세서 import를 깨뜨리고, 오디오는 쓰지 않습니다.
- **버전 고정**: transformers 5.17.0 · accelerate 1.15.0 · peft 0.21.0 · flash-linear-attention 0.5.2
  - Colab stage2와 같은 조합입니다.
  - 설치하는 동안 torch 버전은 고정해 둡니다.
- `7z`가 없으면 apt로 설치합니다(대회 zip 풀기용).
- 마지막에 새 프로세스에서 import를 점검합니다.


In [ ]:
import subprocess, sys, json, shutil, os
def _run(cmd, check=True, quiet=False):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if not quiet or r.returncode: print(f"$ {cmd}\n{(r.stdout or '')[-1200:]}{(r.stderr or '')[-1500:]}")
    if check and r.returncode: raise RuntimeError(f"명령 실패 (코드 {r.returncode}): {cmd} → 위 출력을 확인하세요")
    return r
_PROBE = ("import json, torch; ok = torch.cuda.is_available(); "
          "x = float((torch.randn(256, 256, device='cuda', dtype=torch.bfloat16) @ torch.randn(256, 256, device='cuda', dtype=torch.bfloat16)).float().abs().sum()) if ok else None; "
          "print(json.dumps({'v': torch.__version__, 'cuda': torch.version.cuda, 'ok': bool(ok and x == x), "
          "'cap': list(torch.cuda.get_device_capability(0)) if ok else None, 'n': torch.cuda.device_count()}))")
def torch_probe():
    r = subprocess.run([sys.executable, "-c", _PROBE], capture_output=True, text=True)
    try: return json.loads(r.stdout.strip().splitlines()[-1])
    except Exception: return {"ok": False, "error": ((r.stderr or "") + (r.stdout or ""))[-600:]}
def cuda_ok(p):
    try: return bool(p.get("ok")) and tuple(int(x) for x in str(p.get("cuda")).split(".")[:2]) >= (12, 9)
    except Exception: return False
_p = torch_probe(); print("지금 torch:", _p)
if not cuda_ok(_p):
    _idx = "cu130" if globals().get("DRIVER_MAJOR", 580) >= 580 else "cu129"
    print(f"→ torch를 CUDA {_idx[2:4]}.{_idx[4:]}용 휠로 새로 설치합니다 (2~5분)")
    _run(f"{sys.executable} -m pip install -q --upgrade torch torchvision --index-url https://download.pytorch.org/whl/{_idx}")
    _p = torch_probe(); print("새 torch:", _p)
    assert cuda_ok(_p), f"torch를 새로 설치한 뒤에도 GPU 점검 실패: {_p} → 위 출력을 확인하세요"
_run(f"{sys.executable} -m pip uninstall -y -q torchaudio", check=False, quiet=True)
_run(f"{sys.executable} -m pip uninstall -y -q torchao", check=False, quiet=True)      # 구버전 torchao가 있으면 peft 0.21이 LoRA를 붙일 때 ImportError
_tv = subprocess.run([sys.executable, "-c", "import torchvision; print(torchvision.__version__)"], capture_output=True, text=True).stdout.strip()
open("/tmp/torch_pin.txt", "w").write(f"torch=={_p['v']}\n" + (f"torchvision=={_tv}\n" if _tv else ""))   # 아래 설치가 torch를 바꾸지 못하게 고정
_fla = globals().get("USE_FLA", True)
_run(f"{sys.executable} -m pip install -q -c /tmp/torch_pin.txt 'transformers==5.17.0' 'accelerate==1.15.0' 'peft==0.21.0' "
     + ("'flash-linear-attention==0.5.2' " if _fla else "") + "'huggingface_hub[hf_xet]' pandas pillow")
if not _fla: _run(f"{sys.executable} -m pip uninstall -y -q flash-linear-attention fla-core", check=False, quiet=True); print("USE_FLA = False → 선형 어텐션 커널을 지웠습니다 (PyTorch 기본 구현, 느림)")
if shutil.which("7z") is None:
    _run("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq p7zip-full unzip", check=False, quiet=True)
print("압축 도구:", shutil.which("7z") or shutil.which("unzip") or "없음 → 3번 셀이 파이썬 zipfile로 풉니다(느림)")
_chk = subprocess.run([sys.executable, "-c",
    "import torch, transformers, peft, huggingface_hub, accelerate; " + ("import fla; " if _fla else "fla = None; ") +
    "from transformers import AutoProcessor, AutoModelForImageTextToText; "
    "from transformers.models.qwen3_5_moe import Qwen3_5MoeForConditionalGeneration; "
    "print('torch', torch.__version__, '(CUDA', str(torch.version.cuda) + ')', '| transformers', transformers.__version__, '| peft', peft.__version__, "
    "'| accelerate', accelerate.__version__, '| fla', getattr(fla, '__version__', '?') if fla else '안 씀', '| hub', huggingface_hub.__version__)"],
    capture_output=True, text=True)
print(_chk.stdout.strip())
assert _chk.returncode == 0, "라이브러리 import 점검 실패 → 위 출력을 확인하세요:\n" + (_chk.stderr or "")[-2500:]
print("✅ 설치 완료")


## 3. 데이터 준비 (HF 비공개 저장소 → `/workspace/data`)
- Colab 브리지 노트북 **A 단계**로 올려 둔 대회 zip을 받아 풉니다.
- `/workspace/ssafy-16-2-ai.zip`이 이미 있으면 받지 않습니다. Jupyter 왼쪽 파일 창에 zip을 직접 끌어다 올려도 됩니다.
- HF 토큰은 1번 셀의 `HF_TOKEN`, pod 환경 변수 `HF_TOKEN` 순으로 찾습니다. 둘 다 없으면 입력창이 뜹니다.


In [ ]:
import os, subprocess, shutil, getpass, time
from huggingface_hub import login, whoami, hf_hub_download
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN", "")
if not HF_TOKEN: HF_TOKEN = getpass.getpass("Hugging Face 토큰(Write 권한, hf_로 시작)을 붙여 넣고 Enter: ").strip()
login(token=HF_TOKEN, add_to_git_credential=False)
HF_USER = whoami(token=HF_TOKEN)["name"]
HF_REPO = HF_REPO or f"{HF_USER}/ssafy-vqa-private"
print(f"Hugging Face 로그인: {HF_USER} | 비공개 저장소: {HF_REPO}")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

def _find_data_dir(root):
    if os.path.exists(os.path.join(root, "train.csv")): return root
    for d, _, fs in os.walk(root):
        if "train.csv" in fs and "test.csv" in fs: return d
    return None
if _find_data_dir(DATA_DIR) is None:
    if not os.path.exists(ZIP_PATH):
        t0 = time.time(); print(f"대회 데이터 받는 중: {HF_REPO}/{HF_ZIP_NAME}")
        try:
            _got = hf_hub_download(HF_REPO, HF_ZIP_NAME, repo_type="dataset", token=HF_TOKEN, local_dir=os.path.dirname(ZIP_PATH))
        except Exception as e:
            raise SystemExit(f"❌ {HF_REPO}에서 {HF_ZIP_NAME}를 받지 못했습니다 ({type(e).__name__}: {str(e)[:200]})\n"
                             "→ Colab 브리지 노트북 A 단계를 먼저 실행했는지 확인하세요. 또는 Jupyter 왼쪽 파일 창에 zip을 /workspace로 직접 올리세요.")
        if os.path.abspath(_got) != os.path.abspath(ZIP_PATH): shutil.move(_got, ZIP_PATH)
        print(f"  받음 {os.path.getsize(ZIP_PATH) / 1e9:.2f}GB ({time.time() - t0:.0f}초)")
    t0 = time.time(); print("압축 해제 중... (몇 분)")
    if shutil.which("7z") or shutil.which("unzip"):
        _cmd = (["7z", "x", "-y", f"-p{ZIP_PASSWORD}", f"-o{DATA_DIR}", ZIP_PATH] if shutil.which("7z")
                else ["unzip", "-q", "-o", "-P", ZIP_PASSWORD, ZIP_PATH, "-d", DATA_DIR])
        _r = subprocess.run(_cmd, capture_output=True, text=True)
        assert _r.returncode == 0, f"압축 해제 실패 ({_cmd[0]}) → 비밀번호·파일을 확인하세요:\n" + (_r.stdout or "")[-600:] + (_r.stderr or "")[-800:]
    else:                                               # 7z·unzip 둘 다 없으면 파이썬 zipfile (느림, ZipCrypto 방식만 가능)
        import zipfile
        with zipfile.ZipFile(ZIP_PATH) as _z: _z.extractall(DATA_DIR, pwd=ZIP_PASSWORD.encode())
    print(f"  완료 ({time.time() - t0:.0f}초)")
_dd = _find_data_dir(DATA_DIR)
assert _dd is not None, f"압축을 풀었는데 {DATA_DIR} 아래에 train.csv가 없습니다 → zip 내용을 확인하세요"
if _dd != DATA_DIR: print(f"train.csv가 하위 폴더에 있어 DATA_DIR을 바꿉니다: {_dd}"); DATA_DIR = _dd
for f in ["train.csv", "test.csv", "sample_submission.csv"]:
    assert os.path.exists(os.path.join(DATA_DIR, f)), f"{f} 가 {DATA_DIR} 에 없습니다"
print("데이터 OK:", DATA_DIR, "| 결과 저장:", OUT_DIR)


## 3-1. 모델 가중치 받기 (약 244GB, 처음 한 번만)
- 볼륨 디스크의 HF 캐시(`/workspace/hf`)에 받습니다. 이미 받았으면 몇 초 만에 끝납니다.
- 끊기면 이 셀을 다시 실행하세요. 받은 데까지는 그대로 두고 이어 받습니다.


In [ ]:
import time, os
from huggingface_hub import snapshot_download
t0 = time.time(); print(f"{MODEL_ID} 가중치 확인·받기 (약 244GB)")
MODEL_DIR = snapshot_download(MODEL_ID, token=HF_TOKEN or None, max_workers=16)
_sz = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(MODEL_DIR) for f in fs)   # 캐시의 파일은 심볼릭 링크 → 실제 크기를 따라감
_nst = sum(f.endswith(".safetensors") for f in os.listdir(MODEL_DIR))
print(f"✅ {MODEL_DIR} | safetensors {_nst}개 · {_sz / 1e9:.0f}GB | {time.time() - t0:.0f}초")


## 4. 공통 import · CSV 로드


In [ ]:
import re, json, time, math, hashlib, random, datetime, platform
import numpy as np, pandas as pd, torch
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def _fix_nvrtc():
    # 일부 Colab 이미지(새 CUDA)에서 GPU 정수 텐서의 prod가 런타임 컴파일(NVRTC) 도중 내장 라이브러리를 못 찾아 실패함.
    # 1) 그 라이브러리를 경로로 직접 불러오고, 2) 그래도 안 되면 정수 prod만 CPU에서 계산하도록 우회 (결과는 완전히 같음)
    if not torch.cuda.is_available(): return
    import glob, ctypes
    def _ok():
        try: torch.tensor([[1, 2, 3]], device="cuda").prod(-1).item(); return True, ""
        except RuntimeError as e: return False, str(e)
    good, msg = _ok()
    if good: return
    need = re.findall(r"libnvrtc-builtins\.so[\d.]*", msg)
    pats = ["/usr/local/lib/python3*/dist-packages/nvidia/**/libnvrtc-builtins.so*", "/usr/local/lib/python3*/site-packages/nvidia/**/libnvrtc-builtins.so*",
            "/usr/local/cuda*/lib64/libnvrtc-builtins.so*", "/usr/lib/x86_64-linux-gnu/libnvrtc-builtins.so*"]
    found = sorted({f for p in pats for f in glob.glob(p, recursive=True)})
    for f in found:
        if not need or any(os.path.basename(f).startswith(n) for n in need):
            try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
            except OSError: pass
    if _ok()[0]: print("✅ NVRTC 내장 라이브러리를 직접 불러와 해결:", need); return
    _orig_prod = torch.Tensor.prod
    def _prod(self, *a, **k):
        if self.is_cuda and not (self.is_floating_point() or self.is_complex()):
            return _orig_prod(self.cpu(), *a, **k).to(self.device)
        return _orig_prod(self, *a, **k)
    torch.Tensor.prod = _prod
    print("⚠️ GPU 정수 prod를 CPU에서 계산하도록 우회 (결과 동일) | 확인:", _ok()[0], "| 찾은 라이브러리:", found or "없음")
_fix_nvrtc()

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub_df   = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
dev_df   = pd.read_csv(os.path.join(DATA_DIR, "dev.csv")) if os.path.exists(os.path.join(DATA_DIR, "dev.csv")) else None
for c in ["question", "a", "b", "c", "d"]:
    train_df[c] = train_df[c].astype(str); test_df[c] = test_df[c].astype(str)
train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower()
assert set(train_df.answer.unique()) <= set("abcd"), train_df.answer.unique()
assert list(test_df.id) == list(sub_df.id), "test.csv 와 sample_submission.csv 의 id 순서가 다릅니다"
print(f"train {len(train_df):,} | test {len(test_df):,} | dev {0 if dev_df is None else len(dev_df):,}")
print("정답 분포:", train_df.answer.value_counts().sort_index().to_dict())

def img_path(rel):            # CSV의 path 컬럼은 DATA_DIR 기준 상대경로
    return os.path.join(DATA_DIR, rel)


## 5. 고정 검증셋 분할 (Stage 1과 동일)


In [ ]:
EXPECTED_SPLIT_MD5 = "df05e4bc19d27ebceeef2570672149c2"

def make_split(df, val_size=VAL_SIZE, seed=SEED):
    rng = np.random.RandomState(seed)
    val_idx = []
    for letter in "abcd":
        idx = np.where(df.answer.values == letter)[0]
        rng.shuffle(idx)
        val_idx.extend(idx[: val_size // 4].tolist())
    val_idx = sorted(val_idx)
    mask = np.zeros(len(df), dtype=bool); mask[val_idx] = True
    return df[~mask].reset_index(drop=True), df[mask].reset_index(drop=True)

tr_df, val_df = make_split(train_df)
split_md5 = hashlib.md5(",".join(val_df.id.tolist()).encode()).hexdigest()
print(f"train {len(tr_df):,} / valid {len(val_df):,} | split md5 = {split_md5}")
print("valid 정답 분포:", val_df.answer.value_counts().sort_index().to_dict())
if split_md5 == EXPECTED_SPLIT_MD5:
    print("✅ 팀 공통 분할과 일치")
else:
    print("⚠️⚠️ 검증 분할이 기준과 다릅니다 → 팀원 간 점수 비교 불가. train.csv 원본/SEED/VAL_SIZE를 확인하세요.")
val_df[["id"]].to_csv(os.path.join(OUT_DIR, "valid_ids.csv"), index=False)


## 6. 프롬프트 (Stage 1과 동일)


In [ ]:
PROMPT_VERSION = "baseline_v1"
SYSTEM_INSTRUCT = ("You are a helpful visual question answering assistant. "
                   "Answer using exactly one letter among a, b, c, or d. No explanation.")
LETTERS = "abcd"

def build_mc_prompt(question, options):
    a, b, c, d = options
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

def build_messages(row, perm=(0, 1, 2, 3)):
    # perm[k] = 화면의 k번째 자리에 놓을 '원래 선지'의 인덱스
    opts = [str(row[LETTERS[j]]) for j in perm]
    user_text = build_mc_prompt(str(row["question"]), opts)
    if globals().get("USE_SYSTEM_ROLE", True):
        return [{"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    # system 역할이 없는 템플릿(일부 Gemma 등): 지시문을 user 메시지 앞에 붙임
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": SYSTEM_INSTRUCT + "\n\n" + user_text}]}]

def load_image(rel):
    im = Image.open(img_path(rel))
    im = ImageOps.exif_transpose(im)      # 촬영 방향(EXIF) 보정 — 없으면 그대로
    return im.convert("RGB")


## 7. 모델 로드 + LoRA 부착
출력의 **'답 바로 앞 프롬프트 끝'**이 `</think>` 뒤인지 확인하세요. 244GB를 GPU에 올리는 데 몇 분 걸립니다.
LoRA는 **언어 모델 쪽 선형층**(어텐션·MLP, Qwen3.5의 선형 어텐션 투영 포함)에만 붙이고 비전 타워는 얼려 둡니다.GPU가 2장 이상이면 층을 나눠 올립니다(`device_map="auto"`).


In [ ]:
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel, set_peft_model_state_dict, load_peft_weights, prepare_model_for_kbit_training
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)

import gc
for _v in ("model", "base", "optimizer", "scheduler", "params", "loss", "enc", "logits", "out", "gnorm"):   # 이전 모델·중단된 학습의 그래프가 GPU에 남아 OOM 나는 것 방지
    if _v in globals(): del globals()[_v]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_NAME = torch.cuda.get_device_name(0); cap = torch.cuda.get_device_capability(0)
    BF16_NATIVE = cap[0] >= 8
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
else:
    GPU_NAME, BF16_NATIVE, VRAM_GB = "cpu", False, 0.0
if DEVICE == "cpu":   DTYPE, use_4bit = torch.float32, False
elif BF16_NATIVE:
    _big = re.search(r"(?<![\d.])(2\d|3\d|7\d)B", str(MODEL_ID)) is not None
    DTYPE, use_4bit = torch.bfloat16, (USE_4BIT is True or (USE_4BIT == "auto" and (VRAM_GB < 30 or _big)))
else:                 DTYPE, use_4bit = torch.float16, (USE_4BIT is not False)
USE_FP16_SCALER = (DEVICE == "cuda" and not BF16_NATIVE)
BS = (8 if VRAM_GB >= 38 else 4 if VRAM_GB >= 20 else 2) if BATCH_SIZE == "auto" else int(BATCH_SIZE)
print(f"GPU: {GPU_NAME} ({VRAM_GB:.0f}GB) | dtype={DTYPE} | 4bit={use_4bit} | 평가 배치={BS} | 학습 배치={MICRO_BS}x{GRAD_ACCUM}")

import shutil
RUN_DIR = os.path.join(OUT_DIR, RUN_NAME + ("_dry" if DRY_RUN else ""))
CKPT_DIR = os.path.join(RUN_DIR, "ckpt_last")
if DRY_RUN and os.path.exists(RUN_DIR): shutil.rmtree(RUN_DIR)          # 점검용 실행은 항상 처음부터
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists(CKPT_DIR) and os.path.exists(os.path.join(CKPT_DIR + "_tmp", "trainer_state.pt")):
    os.replace(CKPT_DIR + "_tmp", CKPT_DIR)                              # 저장 직후 교체 단계에서 끊긴 경우 복구

load_kw = dict(dtype=DTYPE, attn_implementation="sdpa")
N_GPU = torch.cuda.device_count() if DEVICE == "cuda" else 0
if DEVICE == "cuda": load_kw["device_map"] = {"": 0} if N_GPU == 1 else "auto"      # 2장 이상(B200 2장 등)이면 층을 나눠 올림
if use_4bit:
    load_kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE,
        llm_int8_skip_modules=["lm_head", ".*visual.*", ".*vision_tower.*", ".*vision_model.*", ".*multi_modal_projector.*", ".*vision_adapter.*", ".*embed_vision.*", ".*audio_tower.*", ".*embed_audio.*"])

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"
tok = processor.tokenizer
_PN = type(processor).__name__
FAMILY = "gemma4" if _PN.startswith("Gemma4") else "muse" if _PN.startswith("MuseGlimmer") else "qwen"   # 프로세서 계열 (해상도 인자·이미지 전달 방식이 다름). EXAONE 4.5는 Qwen2-VL식 → "qwen"
IMAGE_TOKEN_ID = getattr(processor, "image_token_id", None)
if IMAGE_TOKEN_ID is None: IMAGE_TOKEN_ID = tok.convert_tokens_to_ids(getattr(processor, "image_token", "<|image_pad|>"))
_tmpl = processor.chat_template if isinstance(processor.chat_template, str) else json.dumps(processor.chat_template)
TEMPLATE_KW = {"enable_thinking": False} if "enable_thinking" in (_tmpl or "") else {}
if "reasoning_strength" in (_tmpl or ""): TEMPLATE_KW["reasoning_strength"] = MUSE_REASONING   # Muse Glimmer: 템플릿 기본값 high
if "strftime_now" in (_tmpl or ""):                  # 오늘 날짜를 넣는 템플릿 → 날짜 고정
    if "current_date" in _tmpl: TEMPLATE_KW["current_date"] = TEMPLATE_DATE
    else: print("⚠️ 이 템플릿은 오늘 날짜를 프롬프트에 넣습니다 → 날짜가 바뀌면 입력이 달라집니다. 위 출력을 확인하세요")
try:
    processor.apply_chat_template(build_messages({"question": "q", "a": "1", "b": "2", "c": "3", "d": "4"}), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW); USE_SYSTEM_ROLE = True
except Exception as _e:
    USE_SYSTEM_ROLE = False; print("system 역할 미지원 템플릿 → 지시문을 user 메시지 앞에 붙입니다")
print(f"계열: {FAMILY} | 이미지 토큰 id: {IMAGE_TOKEN_ID} | chat template 추가 인자: {TEMPLATE_KW} | system 역할: {USE_SYSTEM_ROLE}")
_probe = processor.apply_chat_template(build_messages({"question": "q", "a": "1", "b": "2", "c": "3", "d": "4"}), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW)
# Muse Glimmer: 생성 프롬프트가 '<|start|>assistant'로 끝나고 다음 토큰에서 추론 채널(' to=self')과 답 채널('<|message|>')이 갈림 → 답 채널을 열어 두고 그다음 글자를 채점
GEN_SUFFIX = ("<|message|>" + MUSE_ANSWER_PREFIX) if (FAMILY == "muse" and _probe.endswith("<|start|>assistant")) else ""
if GEN_SUFFIX_OVERRIDE is not None: GEN_SUFFIX = GEN_SUFFIX_OVERRIDE; print(f"답 자리 앞 문자열을 직접 지정: {GEN_SUFFIX!r}")
if FAMILY == "muse": assert GEN_SUFFIX or _probe.endswith("<|message|>"), f"Muse 템플릿의 생성 프롬프트 끝이 예상과 다릅니다: {_probe[-80:]!r} → 위 출력을 확인하세요"
MUSE_MAX_TOKENS = int(getattr(processor.image_processor, "max_image_tokens", None) or 4096) if FAMILY == "muse" else None   # 모델 한도 (이미지당 병합 토큰)
print("답 바로 앞 프롬프트 끝:", repr((_probe + GEN_SUFFIX)[-80:]))
# 템플릿이 이미 BOS로 시작하면 토크나이저가 또 붙이지 않게 (Muse: 진단에서 BOS 2개 확인. 공식 apply_chat_template도 이렇게 함). 기존 계열은 학습 때와 같게 그대로 둠
NO_DOUBLE_BOS = FAMILY == "muse" and bool(tok.bos_token) and _probe.startswith(tok.bos_token)
import hashlib as _hl
FORMAT_TAG = _hl.md5((_probe + GEN_SUFFIX + str(NO_DOUBLE_BOS)).encode()).hexdigest()[:6]   # 프롬프트 형식이 바뀌면 zero-shot 결과를 새로 잼
ZS_TAG = "zeroshot" if (FAMILY != "muse" and GEN_SUFFIX_OVERRIDE is None) else f"zeroshot_{FORMAT_TAG}"
print(f"BOS 중복 방지: {NO_DOUBLE_BOS} | 형식 태그 {FORMAT_TAG} | zero-shot 파일 태그 {ZS_TAG}")
t0 = time.time()
base = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **load_kw)
base.config.use_cache = False
if use_4bit and not BF16_NATIVE:
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
else:
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
MODEL_LOAD_S = time.time() - t0
print(f"모델 로드 {MODEL_LOAD_S:.0f}s | {type(base).__name__}")
if DEVICE == "cuda":
    print("가중치를 올린 뒤 GPU 메모리: " + " · ".join(f"GPU{i} {torch.cuda.memory_allocated(i) / 2**30:.0f}/{torch.cuda.get_device_properties(i).total_memory / 2**30:.0f}GiB" for i in range(N_GPU)))

LORA_TARGETS = r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|in_proj_qkv|in_proj_z|out_proj)"
RESUMING = os.path.exists(os.path.join(CKPT_DIR, "trainer_state.pt"))
if RESUMING:
    model = PeftModel.from_pretrained(base, os.path.join(CKPT_DIR, "adapter"), is_trainable=True)
    print("↩️ 체크포인트에서 어댑터를 불러왔습니다:", CKPT_DIR)
else:
    torch.manual_seed(TRAIN_SEED)                                        # LoRA 초기화 시드
    model = get_peft_model(base, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                                            target_modules=LORA_TARGETS))
# PEFT는 체크포인팅이 켜져 있으면 '입력 임베딩 출력에 grad 요구' 훅을 거는데, transformers 5.x에서는 이 훅이 비전 타워에도 걸려
# 얼려 둔 비전 타워 전체가 역전파 그래프에 들어갑니다(메모리·시간 낭비). 비재진입(non-reentrant) 체크포인팅에서는 필요 없으므로 제거합니다.
if hasattr(base, "disable_input_require_grads") and getattr(base, "_require_grads_hooks", None):
    base.disable_input_require_grads(); print("입력-grad 훅 제거 (비전 타워는 역전파에서 제외)")
if "qwen3_5" in str(getattr(base.config, "model_type", "")):
    import importlib.util
    if importlib.util.find_spec("fla") is None:
        print("USE_FLA = False → 선형 어텐션을 PyTorch 기본 구현으로 돌립니다 (느림)" if not USE_FLA else "⚠️⚠️ flash-linear-attention이 없습니다 → 선형 어텐션이 순수 PyTorch로 돌아 OOM/저속 위험. 2번 셀을 다시 실행한 뒤 Kernel → Restart Kernel and Run All Cells")
    else:
        print("✅ flash-linear-attention 커널 사용")
model.print_trainable_parameters()
n_lora = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A"))
n_lora_vis = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A") and any(k in n for k in ("visual", "vision_tower", "vision_model")))
print(f"LoRA가 붙은 층: {n_lora}개 (비전 타워 {n_lora_vis}개 — 0이어야 정상)")
assert n_lora > 0 and n_lora_vis == 0

LETTER_IDS = []                                     # 첫 번째(소문자·공백 없음)가 학습 목표, 나머지 변형은 확률 집계용
for L in LETTERS:
    ids = []
    for s in (L, L.upper(), " " + L, " " + L.upper()):
        t = tok.encode(s, add_special_tokens=False)
        if len(t) == 1 and t[0] not in ids: ids.append(t[0])
    assert ids and len(tok.encode(L, add_special_tokens=False)) == 1, f"'{L}' 가 단일 토큰이 아닙니다"
    LETTER_IDS.append(ids)
print("글자 토큰 id:", dict(zip(LETTERS, LETTER_IDS)))


## 8. 채점 함수 (Stage 1과 동일한 로짓 비교) + 순열 TTA
`TTA_PERMS`는 선지를 한 칸씩 밀어 보여주는 4가지 순서입니다. 네 번 채점해 확률을 평균하면 "자리 때문에 생긴 흔들림"이 상쇄됩니다.
진행 상황은 `RUN_DIR/progress.log`에도 적습니다(브라우저를 닫았다 다시 열었을 때 확인용).


In [ ]:
def _plog(msg):                                  # 화면 출력 + RUN_DIR/progress.log (브라우저를 닫아도 진행 상황이 남음)
    print(msg, flush=True)
    try:
        with open(os.path.join(RUN_DIR, "progress.log"), "a", encoding="utf-8") as _f: _f.write(time.strftime("%m-%d %H:%M:%S ") + str(msg) + "\n")
    except Exception: pass
EVAL_TIMES = {}                                  # (해상도, TTA) → [초, 문항 수] — 첫 배치(커널 준비 시간)는 뺌. DRY_RUN 끝 예상 시간 계산용
IDENTITY = (0, 1, 2, 3)
TTA_PERMS = [(0, 1, 2, 3), (1, 2, 3, 0), (2, 3, 0, 1), (3, 0, 1, 2)]

def set_resolution(preset):
    global RES_PRESET, PIXEL_KW, MUSE_PIX
    RES_PRESET = preset
    if FAMILY == "gemma4":
        PIXEL_KW = {"max_soft_tokens": int(RES_PRESETS_GEMMA4[preset])}
    elif FAMILY == "muse":                       # 프로세서는 확대하지 않고 토큰 상한만 둠 → fit_pixels로 Qwen과 같은 픽셀 범위에 맞춰 넘김
        mn, mx = RES_PRESETS[preset]
        mx = min(mx, MUSE_MAX_TOKENS * 28 * 28)  # 모델 한도 4,096토큰 ≈ 321만 픽셀 (up2.0의 상한 340만보다 조금 작음)
        MUSE_PIX = (int(min(mn, mx)), int(mx))
        PIXEL_KW = {"max_image_tokens": int(MUSE_MAX_TOKENS)}
    else:
        mn, mx = RES_PRESETS[preset]
        PIXEL_KW = {"min_pixels": int(mn), "max_pixels": int(mx)}
set_resolution(TRAIN_PRESET)

def render(row, perm=IDENTITY):
    return processor.apply_chat_template(build_messages(row, perm), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW) + GEN_SUFFIX

# 생성으로 보는 zero-shot: 답 자리 첫 토큰이 글자가 아닌 모델(letter_mass가 낮음)도 실력을 잴 수 있게, 그리디로 몇 토큰 생성해 첫 a~d를 읽음
_ANS_RE1 = re.compile(r"^[\s\*\(\[\x22'「『“]*(?:정?답\s*(?:은|는)?\s*[:：]?\s*|(?:the\s+)?(?:correct\s+)?answer\s*(?:is)?\s*[:：]?\s*)?[\(\[\*]*([abcd])(?![a-z])", re.I)
_ANS_RE2 = re.compile(r"[\(\[]([abcd])[\)\]]", re.I)
def parse_answer_letter(s):
    s = str(s).strip()
    m = _ANS_RE1.match(s) or _ANS_RE2.search(s)
    return m.group(1).lower() if m else None

@torch.inference_mode()
def zero_shot_generate(df, max_new_tokens=16):
    was = model.training; model.eval(); outs = []; rows = df.to_dict("records")
    for s in range(0, len(rows), BS):
        b = rows[s: s + BS]
        enc = encode_batch(b, [IDENTITY] * len(b)).to(model.device)
        with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
        outs += [tok.decode(q[enc["input_ids"].shape[1]:], skip_special_tokens=True) for q in out]
    if was: model.train()
    res = df[["id", "answer"]].copy().reset_index(drop=True); res["gen"] = outs; res["pred"] = res.gen.map(parse_answer_letter)
    return res

def fit_pixels(im, min_pixels, max_pixels, factor=28):
    # Qwen2-VL의 smart_resize와 같은 규칙: 가로·세로를 factor 배수로, 전체 픽셀을 [min, max] 안으로 (비율 유지)
    w, h = im.size
    hb, wb = max(factor, round(h / factor) * factor), max(factor, round(w / factor) * factor)
    if hb * wb > max_pixels:
        beta = math.sqrt(h * w / max_pixels)
        hb, wb = max(factor, math.floor(h / beta / factor) * factor), max(factor, math.floor(w / beta / factor) * factor)
    elif hb * wb < min_pixels:
        beta = math.sqrt(min_pixels / (h * w))
        hb, wb = math.ceil(h * beta / factor) * factor, math.ceil(w * beta / factor) * factor
        if hb * wb > max_pixels:                 # 범위가 아주 좁을 때 올림 때문에 상한을 넘으면 상한 기준으로 내림 (실제 프리셋에서는 생기지 않음)
            beta = math.sqrt(h * w / max_pixels)
            hb, wb = max(factor, math.floor(h / beta / factor) * factor), max(factor, math.floor(w / beta / factor) * factor)
    return im if (wb, hb) == (w, h) else im.resize((wb, hb), Image.LANCZOS)

from PIL import ImageEnhance
_AUG_ON = False                                   # train_micro_step 안에서만 True
AUG_RNG = np.random.RandomState(TRAIN_SEED + 12345)
def augment_image(im, rng=None):
    # 글자를 읽는 과제라 기하 변형(반전·회전·자르기)은 쓰지 않고, 촬영 조건에 해당하는 밝기·대비·채도·감마·JPEG 화질만 흔듦
    rng = rng or AUG_RNG
    im = ImageEnhance.Brightness(im).enhance(rng.uniform(0.8, 1.2))
    im = ImageEnhance.Contrast(im).enhance(rng.uniform(0.8, 1.2))
    im = ImageEnhance.Color(im).enhance(rng.uniform(0.9, 1.1))
    g = rng.uniform(0.85, 1.18); im = im.point(lambda v, g=g: int(255 * (v / 255) ** g))
    if rng.rand() < 0.5:
        import io; buf = io.BytesIO(); im.save(buf, "JPEG", quality=int(rng.uniform(60, 95))); buf.seek(0); im = Image.open(buf).convert("RGB")
    return im

def encode_batch(rows, perms):
    texts  = [render(r, p) for r, p in zip(rows, perms)]
    images = [load_image(r["path"]) for r in rows]
    if _AUG_ON and IMG_AUG: images = [augment_image(im) for im in images]
    if FAMILY == "gemma4": images = [[im] for im in images]          # Gemma 프로세서는 샘플별 이미지 리스트를 받음
    if FAMILY == "muse": images = [fit_pixels(im, *MUSE_PIX) for im in images]   # 28 배수 크기로 맞춰 두면 프로세서가 크기를 더 바꾸지 않음
    return processor(text=texts, images=images, padding=True, return_tensors="pt", **PIXEL_KW, **({"add_special_tokens": False} if NO_DOUBLE_BOS else {}))

def _last_logits(enc):
    enc = enc.to(model.device)
    with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        try:    out = model(**enc, logits_to_keep=1, use_cache=False)
        except TypeError: out = model(**enc)
    return out.logits[:, -1, :].float()

def unpermute(prob_slot, perms):
    probs = np.zeros_like(prob_slot)
    for i, p in enumerate(perms):
        for k, j in enumerate(p): probs[i, j] = prob_slot[i, k]
    return probs

@torch.inference_mode()
def score_rows(rows, perms=None):
    global BS
    perms = perms or [IDENTITY] * len(rows)
    enc = logits = None; oom = False
    try:
        enc = encode_batch(rows, perms)
        n_img_tok = (enc["input_ids"] == IMAGE_TOKEN_ID).sum(dim=1).tolist()
        logits = _last_logits(enc)
        if not torch.isfinite(logits).all(): raise FloatingPointError("로짓에 NaN/Inf 발생")
    except torch.cuda.OutOfMemoryError:
        oom = True
    if oom:
        del enc, logits; torch.cuda.empty_cache()
        if len(rows) == 1: raise RuntimeError("배치 1에서도 GPU 메모리 부족")
        h = len(rows) // 2; BS = max(1, min(BS, h)); print(f"  ⚠️ OOM → 평가 배치를 {BS}(으)로 줄입니다")
        p1, m1, t1 = score_rows(rows[:h], perms[:h]); p2, m2, t2 = score_rows(rows[h:], perms[h:])
        return np.concatenate([p1, p2]), np.concatenate([m1, m2]), t1 + t2
    logp = torch.log_softmax(logits, dim=-1)
    slot = torch.stack([torch.logsumexp(logp[:, ids], dim=1) for ids in LETTER_IDS], dim=1)
    mass = slot.exp().sum(dim=1).cpu().numpy()
    return unpermute(torch.softmax(slot, dim=1).cpu().numpy().astype(np.float64), perms), mass, n_img_tok

def score_rows_tta(rows, tta):
    ps, ms, nt = [], [], None
    for perm in (TTA_PERMS if tta else [IDENTITY]):
        p, m, nt = score_rows(rows, [perm] * len(rows)); ps.append(p); ms.append(m)
    return np.mean(ps, axis=0), np.mean(ms, axis=0), nt

def predict_df(df, tag, tta=False, resume=True, verbose=True, save_every=25):
    # df 전체를 채점. resume=True면 CSV에 이어 쓰며, 같은 tag로 다시 부르면 남은 문항만 처리
    was_training = model.training; model.eval()
    out_path = os.path.join(RUN_DIR, f"preds_{tag}__{RES_PRESET}{'__tta4' if tta else ''}__{len(df)}.csv")
    done_ids = set(pd.read_csv(out_path).id) if (resume and os.path.exists(out_path)) else set()
    if not resume and os.path.exists(out_path): os.remove(out_path)
    todo = df[~df.id.isin(done_ids)]
    if SKIP_MISSING_IMAGES: todo = todo[todo.path.map(lambda p: os.path.exists(img_path(p)))]
    rows = todo.to_dict("records"); recs, t0, n_done, nb = [], time.time(), 0, 0
    _tb = []                                        # 배치마다 (걸린 초, 문항 수)
    if verbose: print(f"[{os.path.basename(out_path)}] 전체 {len(df):,} | 완료 {len(done_ids):,} | 남음 {len(rows):,}")
    while n_done < len(rows):
        batch = rows[n_done: n_done + BS]; _t1 = time.time()
        probs, mass, ntok = score_rows_tta(batch, tta)
        _tb.append((time.time() - _t1, len(batch)))
        for r, p, m, t in zip(batch, probs, mass, ntok):
            recs.append({"id": r["id"], "pred": LETTERS[int(p.argmax())], "p_a": p[0], "p_b": p[1], "p_c": p[2], "p_d": p[3],
                         "letter_mass": float(m), "img_tokens": int(t)})
        n_done += len(batch); nb += 1
        if nb % save_every == 0 or n_done >= len(rows):
            pd.DataFrame(recs).to_csv(out_path, mode="a", header=not os.path.exists(out_path), index=False); recs = []
            if verbose:
                el = time.time() - t0
                _plog(f"  {os.path.basename(out_path)} {n_done:>5}/{len(rows)} | {el / n_done:.2f}s/문항 | 남은 시간 ~{(len(rows) - n_done) * el / n_done / 60:.0f}분")
    if _tb:
        _use = _tb[1:] if len(_tb) > 1 else _tb
        _e = EVAL_TIMES.setdefault((RES_PRESET, bool(tta)), [0.0, 0]); _e[0] += sum(t for t, _ in _use); _e[1] += sum(n for _, n in _use)
    if was_training: model.train()
    return pd.read_csv(out_path).drop_duplicates("id", keep="last"), out_path

CATS = {"전화번호": r"전화|번호|연락처", "가격/금액": r"가격|얼마|금액|요금|할인", "이름/상호/제목": r"이름|상호|제목|브랜드|가게|매장|간판|업체",
        "날짜/시간": r"날짜|언제|시간|기간|요일|몇 시|년도|연도", "위치/방향/층": r"어디|위치|방향|몇 층|층에|왼쪽|오른쪽|위쪽|아래",
        "개수/숫자": r"몇 개|몇 명|몇 번|개수|몇 가지|숫자", "색상": r"색|컬러", "부정형": r"아닌|않는|없는|틀린|잘못"}

def evaluate(df, tag, tta=False, resume=True, verbose=True, by_type=True):
    pred, _ = predict_df(df, tag, tta=tta, resume=resume, verbose=verbose)
    m = df.merge(pred, on="id", how="inner"); m["correct"] = (m.pred == m.answer)
    acc = float(m.correct.mean()); n = len(m); se = math.sqrt(max(acc * (1 - acc), 1e-9) / n)
    _plog(f"▶ {tag} | {RES_PRESET}{' +TTA4' if tta else ''} | 정확도 {acc * 100:.2f}% (n={n}, ±{1.96 * se * 100:.1f}%p) | letter_mass {m.letter_mass.mean():.3f}")
    if by_type:
        for k, pat in CATS.items():
            sel = m.question.str.contains(pat)
            if sel.sum() >= 10: print(f"    {k:<12} {m.correct[sel].mean() * 100:5.1f}%  (n={int(sel.sum())})")
    lp = os.path.join(OUT_DIR, "results_log_s2.csv")
    log = {"time": datetime.datetime.now().strftime("%m-%d %H:%M"), "run": f"{RUN_NAME}{'_dry' if DRY_RUN else ''}/{tag}", "model": str(MODEL_ID),
           "res": RES_PRESET, "tta": int(tta), "n": n, "acc": round(acc * 100, 2), "ci95": round(1.96 * se * 100, 2),
           "letter_mass": round(float(m.letter_mass.mean()), 3), "gpu": GPU_NAME, "split_md5": split_md5[:8]}
    pd.DataFrame([log]).to_csv(lp, mode="a", header=not os.path.exists(lp), index=False)
    return acc, m


## 9. 학습 준비: 배치 구성 · 손실 · 체크포인트
- `epoch_plan`: epoch마다 문항 순서와 선지 순열을 **시드로 결정** → 중간에 끊겼다 이어도 똑같은 데이터 순서가 재현됩니다.
- 손실: 추론과 같은 프롬프트의 **마지막 위치 로짓**에, 섞인 순서 기준의 정답 글자(소문자 토큰)로 교차 엔트로피.


In [ ]:
train_src = train_df if TRAIN_ON_FULL else tr_df
if SKIP_MISSING_IMAGES: train_src = train_src[train_src.path.map(lambda p: os.path.exists(img_path(p)))]
if DRY_RUN:       train_src = train_src.head(3 * MICRO_BS * GRAD_ACCUM)
elif TRAIN_LIMIT: train_src = train_src.head(int(TRAIN_LIMIT))
if EXTRA_TRAIN_CSV:                                   # 추가 학습 데이터(예: dev 의사 라벨)는 위 제한과 별개로 뒤에 붙임
    _need = ["id", "path", "question", "a", "b", "c", "d", "answer"]
    extra = pd.read_csv(EXTRA_TRAIN_CSV); assert all(c in extra.columns for c in _need), f"추가 데이터 열이 부족합니다: {list(extra.columns)}"
    for c in ["question", "a", "b", "c", "d"]: extra[c] = extra[c].astype(str)
    extra["answer"] = extra["answer"].astype(str).str.strip().str.lower(); assert set(extra.answer.unique()) <= set("abcd"), extra.answer.unique()
    assert not (set(extra.id) & set(val_df.id)), "추가 데이터에 검증 문항이 섞여 있습니다"
    assert extra.path.map(lambda p: os.path.exists(img_path(p))).all(), "추가 데이터의 이미지 경로가 없습니다"
    if DRY_RUN: extra = extra.head(2)
    train_src = pd.concat([train_src[_need], extra[_need]], ignore_index=True)
    print(f"추가 학습 데이터 {len(extra):,}문항 합침 (정답 분포 {extra.answer.value_counts().sort_index().to_dict()}) → 학습 총 {len(train_src):,}문항")
train_rows = train_src.to_dict("records")
STEP_SAMPLES = MICRO_BS * GRAD_ACCUM
steps_per_epoch = len(train_rows) // STEP_SAMPLES
n_epochs = 1 if DRY_RUN else EPOCHS
total_steps = steps_per_epoch * n_epochs
if MAX_STEPS and not DRY_RUN: total_steps = min(total_steps, int(MAX_STEPS))
assert total_steps > 0, "학습 샘플이 유효 배치보다 적습니다"
print(f"학습 문항 {len(train_rows):,} | 유효 배치 {STEP_SAMPLES} | epoch당 {steps_per_epoch} 스텝 | 총 {total_steps} 스텝" + (" | ⚠️ 검증셋 포함 학습(TRAIN_ON_FULL)" if TRAIN_ON_FULL else ""))

def epoch_plan(epoch):
    rng = np.random.RandomState(TRAIN_SEED * 1000 + epoch)
    order = rng.permutation(len(train_rows))
    perms = [tuple(rng.permutation(4).tolist()) if SHUFFLE_OPTIONS else IDENTITY for _ in range(len(train_rows))]
    return order, perms

def gold_slot(row, perm):                 # 섞인 화면에서 정답이 놓인 자리 (0~3)
    return perm.index(LETTERS.index(row["answer"]))

def train_micro_step(rows, perms):
    global _AUG_ON
    _AUG_ON = True
    try: enc = encode_batch(rows, perms)
    finally: _AUG_ON = False
    target = torch.tensor([LETTER_IDS[gold_slot(r, p)][0] for r, p in zip(rows, perms)], device=model.device)
    logits = _last_logits(enc)
    loss = F.cross_entropy(logits, target.to(logits.device))       # GPU 여러 장이면 로짓이 마지막 GPU에 있음
    with torch.no_grad():
        slot = torch.stack([logits[:, ids[0]] for ids in LETTER_IDS], dim=1)
        hit = (slot.argmax(1).cpu() == torch.tensor([gold_slot(r, p) for r, p in zip(rows, perms)])).float().mean().item()
    return loss, hit

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, int(total_steps * WARMUP_RATIO)), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16_SCALER)
state = {"global_step": 0, "history": [], "epoch_acc": {}, "config": {k: globals()[k] for k in
         ["MODEL_ID", "TRAIN_PRESET", "EPOCHS", "MAX_STEPS", "LR", "STEP_SAMPLES", "LORA_R", "LORA_ALPHA", "SHUFFLE_OPTIONS", "TRAIN_ON_FULL", "SEED", "TRAIN_SEED", "EXTRA_TRAIN_CSV", "IMG_AUG", "TRAIN_PRESETS"]}}

def save_checkpoint():
    tmp = CKPT_DIR + "_tmp"; os.makedirs(tmp, exist_ok=True)
    model.save_pretrained(os.path.join(tmp, "adapter"))
    rng = {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state(),
           "cuda": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None}
    torch.save({"optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "state": state, "rng": rng}, os.path.join(tmp, "trainer_state.pt"))
    if os.path.exists(CKPT_DIR): shutil.rmtree(CKPT_DIR)
    os.replace(tmp, CKPT_DIR)                     # 저장 도중 끊겨도 이전 체크포인트가 깨지지 않도록 교체 방식으로

if RESUMING:
    ck = torch.load(os.path.join(CKPT_DIR, "trainer_state.pt"), map_location="cpu", weights_only=False)
    _defaults = {"TRAIN_SEED": 42, "MAX_STEPS": None, "EXTRA_TRAIN_CSV": None, "IMG_AUG": False, "TRAIN_PRESETS": None}
    same = all(ck["state"]["config"].get(k, _defaults.get(k)) == v for k, v in state["config"].items())
    assert same, f"체크포인트의 설정이 현재와 다릅니다. RUN_NAME을 바꾸거나 설정을 되돌리세요.\n저장됨: {ck['state']['config']}\n현재: {state['config']}"
    optimizer.load_state_dict(ck["optimizer"]); scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    state = ck["state"]
    random.setstate(ck["rng"]["py"]); np.random.set_state(ck["rng"]["np"]); torch.set_rng_state(ck["rng"]["torch"])
    if DEVICE == "cuda" and ck["rng"]["cuda"] is not None: torch.cuda.set_rng_state_all(ck["rng"]["cuda"])
    print(f"↩️ {state['global_step']}/{total_steps} 스텝부터 이어서 학습합니다")


## 10. 학습 루프 (+ 학습 전 zero-shot 점검)
zero-shot 점검은 LoRA가 아직 0인 상태(= 기본 모델)로 채점합니다. Drive에 다른 실행의 zero-shot 예측(`preds_zeroshot*.csv`, 예: E5 폴더의 Q35)이 있으면 같은 문항 기준 정확도를 같이 보여 줍니다.
로그의 `loss`는 정답 글자에 대한 교차 엔트로피(처음엔 0.2 안팎에서 시작해 내려가야 정상), `train_acc`는 학습 배치에서의 정답률입니다.
**122B 전용 안전장치**
- 본 학습(`DRY_RUN = False`)에서 zero-shot이 `ZERO_SHOT_MIN_ACC`(93%)보다 낮으면 학습 전에 멈춥니다. 무언가 고장 난 상태에서 비싼 학습을 하지 않기 위해서입니다.
- DRY_RUN 끝에는 실제로 잰 속도로 **본 실행 전체의 예상 시간·비용 표**를 보여 줍니다.


In [ ]:
_STOP_AT = None      # (테스트용) 이 스텝에서 강제 중단
quick_df = val_df.head(QUICK_EVAL_N if not DRY_RUN else 3 * BS)
if SKIP_MISSING_IMAGES: quick_df = quick_df[quick_df.path.map(lambda p: os.path.exists(img_path(p)))]
if ZERO_SHOT_N and state["global_step"] == 0 and not RESUMING:
    set_resolution(ZERO_SHOT_PRESET)
    zs_df = val_df.head(int(ZERO_SHOT_N) if not DRY_RUN else 3 * BS)
    if SKIP_MISSING_IMAGES: zs_df = zs_df[zs_df.path.map(lambda p: os.path.exists(img_path(p)))]
    print(f"학습 전 zero-shot 점검: 검증 앞 {len(zs_df)}문항 | 해상도 {ZERO_SHOT_PRESET} | LoRA가 아직 0이라 기본 모델과 같음")
    zs_acc, zs_m = evaluate(zs_df, ZS_TAG, resume=True, verbose=False, by_type=False)
    zs_mass = float(zs_m.letter_mass.mean())
    state["history"].append({"step": 0, "zero_shot_acc": zs_acc, "zero_shot_n": len(zs_m), "zero_shot_mass": zs_mass})
    import glob as _glob
    for _f in sorted(_glob.glob(os.path.join(OUT_DIR, "*", "preds_zeroshot*.csv"))):      # 같은 문항에서 다른 실행의 zero-shot 기록과 비교
        if os.path.abspath(os.path.dirname(_f)) == os.path.abspath(RUN_DIR): continue
        try: _r = zs_m[["id", "answer"]].merge(pd.read_csv(_f, usecols=["id", "pred"]).drop_duplicates("id", keep="last"), on="id")
        except Exception: continue
        if len(_r) == len(zs_m): print(f"    같은 {len(zs_m)}문항 기준 {os.path.basename(os.path.dirname(_f))}/{os.path.basename(_f)}: {(_r.pred == _r.answer).mean() * 100:.2f}%")
    print("    참고 기록: Q35 제로샷 96.1% → 학습 후 97.2% (검증 1,000문항, up2.0) · Gemma4-31B 제로샷 97.67% → 학습 후 98.33% (검증 앞 300문항)")
    print("    대략의 기준: 같은 문항에서 Q35 제로샷보다 3%p 넘게 낮으면 학습 후에도 96%대 이하일 가능성이 큼 (Q35·Gemma의 학습 이득은 +0.7~1.1%p였음)")
    if zs_mass < 0.9:
        print(f"    ⚠️ letter_mass {zs_mass:.3f}: 답 자리에서 a~d가 아닌 토큰을 내려는 확률이 큼 → 위 정확도는 글자 확률이 거의 0인 상태의 잡음이라 실력이 아닙니다. "
              "학습하면 이 자리에서 바로 글자를 내도록 배우므로 학습 자체에는 문제없음")
    if zs_mass < 0.5:
        with torch.inference_mode():
            _pr = torch.softmax(_last_logits(encode_batch(zs_df.head(2).to_dict("records"), [IDENTITY] * 2)), dim=-1)
        for _i, _r in enumerate(zs_df.head(2).to_dict("records")):
            _tp = torch.topk(_pr[_i], 8)
            print(f"    답 자리 상위 토큰 ({_r['id']}, 정답 {_r['answer']}): " + ", ".join(f"{tok.decode([int(t)])!r} {float(p):.3f}" for p, t in zip(_tp.values, _tp.indices)))
        _gfp = os.path.join(RUN_DIR, f"preds_{ZS_TAG}_gen__{ZERO_SHOT_PRESET}__{len(zs_df)}.csv")
        if os.path.exists(_gfp) and list(pd.read_csv(_gfp).id) == list(zs_df.id):
            zs_gen = pd.read_csv(_gfp, keep_default_na=False); zs_gen["pred"] = zs_gen.gen.map(parse_answer_letter)
        else:
            print(f"    → 생성으로 다시 잽니다: 그리디 16토큰 → 첫 a~d 글자 ({len(zs_df)}문항)")
            zs_gen = zero_shot_generate(zs_df); zs_gen.to_csv(_gfp, index=False)
        for _r in zs_gen.head(6).itertuples(): print(f"      {_r.id} 정답 {_r.answer} → 생성 {_r.gen!r} → {_r.pred}")
        zs_gen_acc = float((zs_gen.pred == zs_gen.answer).mean()); _parsed = float(zs_gen.pred.notna().mean())
        state["history"][-1].update({"zero_shot_gen_acc": zs_gen_acc, "zero_shot_gen_parsed": _parsed})
        print(f"    ▶ 생성 기반 zero-shot: {zs_gen_acc * 100:.2f}% (n={len(zs_gen)}, 글자를 읽어 낸 비율 {_parsed * 100:.0f}%) ← 위 다른 실행들의 zero-shot과 비교할 값")
    set_resolution(TRAIN_PRESET)
    if not DRY_RUN and ZERO_SHOT_MIN_ACC and zs_acc < ZERO_SHOT_MIN_ACC:
        raise SystemExit(f"❌ zero-shot {zs_acc * 100:.2f}% < 기준 {ZERO_SHOT_MIN_ACC * 100:.0f}% → 프롬프트·해상도·커널 중 무언가가 고장 난 것으로 보고 학습 전에 멈춥니다 (비용 절약). "
                         "이 셀의 출력을 확인하세요.")
    if ZERO_SHOT_ONLY:
        raise SystemExit("ZERO_SHOT_ONLY=True → 학습하지 않고 여기서 멈춥니다. 학습하려면 1번 셀에서 False로 바꾸고 처음부터 다시 실행하세요 (이 zero-shot 결과는 재사용됩니다).")
if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
model.train(); optimizer.zero_grad(set_to_none=True)
t_start, run_loss, run_hit, n_micro = time.time(), 0.0, 0.0, 0
STEP_TIMES = []                                    # 옵티마이저 스텝마다 학습에 걸린 초 (평가·저장 제외)
step0 = state["global_step"]; plan_epoch = -1

for step in range(step0, total_steps):
    epoch, k = divmod(step, steps_per_epoch)
    if epoch != plan_epoch: order, perms_all = epoch_plan(epoch); plan_epoch = epoch
    idx = order[k * STEP_SAMPLES: (k + 1) * STEP_SAMPLES]
    _ts = time.time()
    while True:                                        # OOM이면 마이크로 배치를 절반으로 줄여 같은 스텝을 다시 시도 (유효 배치는 그대로)
        oom = False; s_loss, s_hit, s_n = 0.0, 0.0, 0
        try:
            for s in range(0, STEP_SAMPLES, MICRO_BS):
                mb = idx[s: s + MICRO_BS]
                if TRAIN_PRESETS: set_resolution(TRAIN_PRESETS[np.random.RandomState(TRAIN_SEED * 100003 + step * 97 + s).randint(len(TRAIN_PRESETS))])   # 결정적 → 재개해도 같음
                loss, hit = train_micro_step([train_rows[i] for i in mb], [perms_all[i] for i in mb])
                scaler.scale(loss / GRAD_ACCUM).backward()
                s_loss += loss.item(); s_hit += hit; s_n += 1
        except torch.cuda.OutOfMemoryError:
            oom = True
        if not oom: break
        loss = None; optimizer.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
        if MICRO_BS == 1: raise RuntimeError("마이크로 배치 1에서도 GPU 메모리 부족 → TRAIN_PRESET을 낮추거나 4bit를 켜세요.")
        MICRO_BS //= 2; GRAD_ACCUM *= 2
        print(f"  ⚠️ OOM → MICRO_BS={MICRO_BS}, GRAD_ACCUM={GRAD_ACCUM} 로 바꿔 이 스텝을 다시 실행합니다", flush=True)
    run_loss += s_loss; run_hit += s_hit; n_micro += s_n
    if TRAIN_PRESETS: set_resolution(TRAIN_PRESET)
    scaler.unscale_(optimizer)
    gnorm = torch.nn.utils.clip_grad_norm_(params, 1.0)
    scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
    state["global_step"] = step + 1
    STEP_TIMES.append(time.time() - _ts)

    if (step + 1) % LOG_EVERY == 0 or step + 1 == total_steps or DRY_RUN:
        el = time.time() - t_start; done = step + 1 - step0
        _plog(f"step {step + 1:>4}/{total_steps} | ep {epoch + 1} | loss {run_loss / n_micro:.4f} | train_acc {run_hit / n_micro * 100:5.1f}% | "
              f"lr {scheduler.get_last_lr()[0]:.2e} | gnorm {float(gnorm):.2f} | {el / done:.1f}s/step | 남은 시간 ~{(total_steps - step - 1) * el / done / 60:.0f}분")
        state["history"].append({"step": step + 1, "loss": run_loss / n_micro, "train_acc": run_hit / n_micro})
        run_loss, run_hit, n_micro = 0.0, 0.0, 0
    end_of_epoch = (step + 1) % steps_per_epoch == 0 or step + 1 == total_steps      # MAX_STEPS로 중간에 끝나는 경우도 'epoch 끝'으로 취급
    if ((step + 1) % EVAL_EVERY == 0 and not end_of_epoch) or (DRY_RUN and step + 1 == total_steps):
        acc, _ = evaluate(quick_df, f"quick_step{step + 1}", resume=False, verbose=False, by_type=False)
        state["history"].append({"step": step + 1, "quick_val_acc": acc})
    if end_of_epoch and not DRY_RUN:
        acc, _ = evaluate(val_df, f"ep{epoch + 1}", resume=False, verbose=False)
        state["epoch_acc"][epoch + 1] = acc
        model.save_pretrained(os.path.join(RUN_DIR, f"adapter_ep{epoch + 1}"))
    if (step + 1) % SAVE_EVERY == 0 or end_of_epoch or step + 1 == total_steps:
        save_checkpoint()
    if _STOP_AT and step + 1 == _STOP_AT: print("강제 중단(테스트)"); break

if DEVICE == "cuda":
    _plog("최대 GPU 메모리: " + " · ".join(f"GPU{i} {torch.cuda.max_memory_allocated(i) / 2**30:.1f}/{torch.cuda.get_device_properties(i).total_memory / 2**30:.0f}GiB" for i in range(N_GPU)))
if DRY_RUN and total_steps > step0:
    _st = STEP_TIMES[1:] if len(STEP_TIMES) > 1 else STEP_TIMES          # 첫 스텝은 커널 준비 시간이 섞여 뺌
    per_step = float(np.median(_st)) if _st else 0.0
    _spe_full = len(train_df if TRAIN_ON_FULL else tr_df) // STEP_SAMPLES   # 본 실행의 epoch당 스텝 (DRY_RUN의 steps_per_epoch는 3)
    full_steps = _spe_full * EPOCHS
    if MAX_STEPS: full_steps = min(full_steps, int(MAX_STEPS))
    def _spq(preset, tta=False):                                        # 문항당 초 (DRY_RUN에서 잰 값, 없으면 가장 비싼 값)
        v = EVAL_TIMES.get((preset, tta)) or EVAL_TIMES.get((preset, False))
        if v and v[1]: return v[0] / v[1] * (4 if (tta and (preset, tta) not in EVAL_TIMES) else 1)
        return max((t / n for t, n in EVAL_TIMES.values() if n), default=0.0)
    _n_quick = sum(1 for s_ in range(1, full_steps + 1) if s_ % EVAL_EVERY == 0 and s_ % _spe_full != 0 and s_ != full_steps)
    _test_preset = TEST_PRESET if TEST_PRESET != "auto" else EVAL_PRESETS[0]
    _rows = [("가중치 GPU에 올리기", globals().get("MODEL_LOAD_S", 0.0)),
             (f"zero-shot {ZERO_SHOT_N}문항 ({ZERO_SHOT_PRESET})", (ZERO_SHOT_N or 0) * _spq(ZERO_SHOT_PRESET)),
             (f"학습 {full_steps}스텝 × {per_step:.1f}초", full_steps * per_step),
             (f"학습 중 빠른 검증 {_n_quick}번 × {QUICK_EVAL_N}문항 ({TRAIN_PRESET})", _n_quick * QUICK_EVAL_N * _spq(TRAIN_PRESET)),
             (f"epoch 끝 검증 {len(val_df)}문항 ({TRAIN_PRESET})", EPOCHS * len(val_df) * _spq(TRAIN_PRESET))]
    _rows += [(f"최종 검증 {len(val_df)}문항 ({p_})", len(val_df) * _spq(p_)) for p_ in EVAL_PRESETS if p_ != TRAIN_PRESET]
    if EVAL_TTA: _rows.append((f"최종 검증 TTA ({EVAL_PRESETS[0]})", len(val_df) * _spq(EVAL_PRESETS[0], True)))
    if MAKE_SUBMISSION: _rows.append((f"test {len(test_df)}문항 ({_test_preset}{' +TTA4' if TEST_TTA else ''})", len(test_df) * _spq(_test_preset, TEST_TTA)))
    _tot = sum(v for _, v in _rows)
    print("\n✅ DRY RUN 통과 — 본 실행(DRY_RUN=False) 예상 (이 GPU에서 잰 속도 기준):")
    for k_, v_ in _rows: print(f"    · {k_}: {v_ / 60:.1f}분")
    print(f"    ▶ 합계 {_tot / 60:.1f}분 ≈ {_tot / 3600:.1f}시간 → 비용 약 ${_tot / 3600 * GPU_PRICE_PER_HOUR:.0f} (시간당 ${GPU_PRICE_PER_HOUR})")
    print("    (설치·데이터·가중치는 이미 끝나 있어 빠짐. 실제로는 ±30% 정도 달라질 수 있음)")
    print("→ 1번 셀에서 DRY_RUN = False로 바꾸고 Kernel → Restart Kernel and Run All Cells")
json.dump({"epoch_acc": state["epoch_acc"], "history": state["history"], "config": state["config"]},
          open(os.path.join(RUN_DIR, "train_log.json"), "w"), indent=1, default=str)


## 11. 최종 평가: 가장 좋은 epoch의 어댑터 → 해상도별 · TTA 유무별 검증 점수 + 오답 리포트
`error_report_*.html`은 틀린 문항을 이미지와 함께 모은 파일입니다. Drive에서 내려받아 브라우저로 열면 되고(런타임 불필요).


In [ ]:
import base64, io, html as _html
def export_error_report(m, path, max_items=150):
    wrong = m[~m.correct].head(max_items); parts = [f"<h2>{_html.escape(os.path.basename(path))} — 오답 {int((~m.correct).sum())}건 / {len(m)}문항</h2>"]
    for _, r in wrong.iterrows():
        im = load_image(r.path); im.thumbnail((560, 560)); buf = io.BytesIO(); im.save(buf, "JPEG", quality=72)
        opts = "".join(f"<li>({k}) {_html.escape(str(r[k]))} — p={r['p_' + k]:.2f}{' <b>← 정답</b>' if k == r.answer else ''}{' <b style=color:#c00>← 예측</b>' if k == r.pred else ''}</li>" for k in LETTERS)
        parts.append(f"<div style='display:flex;gap:16px;margin:18px 0;border-top:1px solid #ccc;padding-top:12px'><img src='data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}'>"
                     f"<div><b>{r.id}</b><p>{_html.escape(str(r.question))}</p><ul>{opts}</ul></div></div>")
    open(path, "w", encoding="utf-8").write("<meta charset='utf-8'><body style='font-family:sans-serif;max-width:1100px;margin:auto'>" + "".join(parts))
    print("오답 리포트 저장:", path)

if not DRY_RUN and state["global_step"] < total_steps:
    raise RuntimeError(f"학습이 아직 끝나지 않았습니다 ({state['global_step']}/{total_steps} 스텝). 1번 셀부터 다시 실행하면 체크포인트에서 이어서 학습합니다.")
if not DRY_RUN and state["epoch_acc"]:
    best_ep = max(state["epoch_acc"], key=lambda e: (state["epoch_acc"][e], e))     # 동률이면 더 오래 학습한 쪽
    print("epoch별 검증 정확도:", {e: round(a * 100, 2) for e, a in state["epoch_acc"].items()}, "→ 사용할 어댑터: epoch", best_ep)
    set_peft_model_state_dict(model, load_peft_weights(os.path.join(RUN_DIR, f"adapter_ep{best_ep}"), device="cpu"))
    FINAL_TAG = f"final_ep{best_ep}"
else:
    FINAL_TAG = "final_dry"
model.eval()
final_eval_df = val_df if not DRY_RUN else quick_df
summary = []
for preset in (EVAL_PRESETS if not DRY_RUN else [TRAIN_PRESET]):
    set_resolution(preset)
    if not DRY_RUN and preset == TRAIN_PRESET:                    # epoch 끝 평가와 같은 모델·해상도 → 그 결과를 그대로 재사용
        _src = os.path.join(RUN_DIR, f"preds_ep{best_ep}__{preset}__{len(final_eval_df)}.csv")
        _dst = os.path.join(RUN_DIR, f"preds_{FINAL_TAG}__{preset}__{len(final_eval_df)}.csv")
        if os.path.exists(_src) and not os.path.exists(_dst): shutil.copy(_src, _dst)
    acc, m = evaluate(final_eval_df, FINAL_TAG, tta=False)
    summary.append({"해상도": preset, "TTA": False, "정확도(%)": round(acc * 100, 2)})
    export_error_report(m, os.path.join(RUN_DIR, f"error_report_{FINAL_TAG}_{preset}.html"))
BEST_PRESET = max(summary, key=lambda s: s["정확도(%)"])["해상도"]        # 동률이면 앞에 적은(=더 싼) 해상도
if EVAL_TTA:
    set_resolution(BEST_PRESET)
    acc, _ = evaluate(final_eval_df, FINAL_TAG, tta=True, by_type=False)
    summary.append({"해상도": BEST_PRESET, "TTA": True, "정확도(%)": round(acc * 100, 2)})
print("\n" + pd.DataFrame(summary).to_string(index=False))
print(f"→ 제출 권장 해상도: {BEST_PRESET}")
if TRAIN_ON_FULL: print("⚠️ TRAIN_ON_FULL=True: 검증셋을 학습에 썼으므로 위 점수는 의미가 없습니다 (제출용 모델).")
set_resolution(TRAIN_PRESET)


## 12. test 추론 → 제출 파일 (이어하기 지원)


In [ ]:
if MAKE_SUBMISSION and not DRY_RUN:
    test_preset = BEST_PRESET if TEST_PRESET == "auto" else TEST_PRESET
    set_resolution(test_preset)
    pred, _ = predict_df(test_df, f"{FINAL_TAG}_test", tta=TEST_TTA)
    sub = sub_df[["id"]].merge(pred[["id", "pred"]], on="id", how="left").rename(columns={"pred": "answer"})
    assert len(sub) == len(sub_df) and list(sub.id) == list(sub_df.id), "id 순서/개수 불일치"
    assert sub.answer.notna().all(), f"예측 누락 {int(sub.answer.isna().sum())}건"
    assert set(sub.answer.unique()) <= set(LETTERS), sub.answer.unique()
    sp = os.path.join(RUN_DIR, f"submission_{RUN_NAME}_{FINAL_TAG}_{test_preset}{'_tta4' if TEST_TTA else ''}.csv"); sub.to_csv(sp, index=False)
    print("저장:", sp, "| 예측 분포:", sub.answer.value_counts().sort_index().to_dict())
    set_resolution(TRAIN_PRESET)


## 13. 결과 요약


In [ ]:
print("RUN:", RUN_NAME, "| MODEL:", MODEL_ID, "| epoch별 검증 정확도:", {e: round(a * 100, 2) for e, a in state["epoch_acc"].items()})
_zs = [x for x in state["history"] if "zero_shot_acc" in x]
if _zs: print(f"학습 전 zero-shot: {_zs[-1]['zero_shot_acc'] * 100:.2f}% (n={_zs[-1]['zero_shot_n']}, letter_mass {_zs[-1]['zero_shot_mass']:.3f})"
              + (f" | 생성 기반 {_zs[-1]['zero_shot_gen_acc'] * 100:.2f}% (글자 읽음 {_zs[-1]['zero_shot_gen_parsed'] * 100:.0f}%)" if "zero_shot_gen_acc" in _zs[-1] else ""))
h = pd.DataFrame([x for x in state["history"] if "loss" in x])
if len(h): print("loss 추이(처음→끝):", [round(v, 4) for v in h.loss.iloc[:: max(1, len(h) // 8)].tolist()])
q = [(x["step"], round(x["quick_val_acc"] * 100, 1)) for x in state["history"] if "quick_val_acc" in x]
if q: print("학습 중 빠른 검증(step, 정확도%):", q)
lp = os.path.join(OUT_DIR, "results_log_s2.csv")
if os.path.exists(lp):
    L = pd.read_csv(lp); L = L[L.run.astype(str).str.startswith(RUN_NAME)]
    print(L[[c for c in ["time", "run", "res", "tta", "n", "acc", "ci95", "letter_mass"] if c in L.columns]].tail(12).to_string(index=False))
if not DRY_RUN: print("\n다음: 14번 셀로 결과를 HF 비공개 저장소에 올리세요 (pod를 끄기 전에)")


## 14. 결과를 HF 비공개 저장소로 올리기 — pod를 끄기 전에 꼭
- 올리기 전에 ens_search가 읽는 파일을 확인합니다.
  - 검증 예측 1,000문항
  - test 예측 6,714문항
  - `train_log.json`
  - 최종 어댑터
- 체크포인트(`ckpt_last`, 이어하기용)는 올리지 않습니다. 최종 어댑터 `adapter_ep*`는 재현성 제출용으로 올립니다.
- 다 올라가면 Colab 브리지 노트북 **B 단계**로 Drive에 받습니다. Drive에 들어온 것을 확인한 뒤 RunPod에서 pod를 **Terminate**하세요.


In [ ]:
import re, glob, shutil, time
from huggingface_hub import HfApi
assert not DRY_RUN, "DRY_RUN 결과는 올리지 않습니다 → DRY_RUN = False로 본 실행을 끝낸 뒤 이 셀을 실행하세요"
_fs = os.listdir(RUN_DIR)
_val = sorted(f for f in _fs if re.match(r"^preds_final_ep\d+__(.+)__" + str(len(val_df)) + r"\.csv$", f))
_tst = sorted(f for f in _fs if re.match(r"^preds_final_ep\d+_test__(.+)__" + str(len(test_df)) + r"\.csv$", f))
assert _val, f"최종 검증 예측 파일이 없습니다 ({RUN_DIR}) → 11번 셀까지 끝났는지 확인하세요"
assert _tst, f"test 예측 파일이 없습니다 ({RUN_DIR}) → 12번 셀까지 끝났는지 확인하세요"
for f, n in [(x, len(val_df)) for x in _val] + [(x, len(test_df)) for x in _tst]:
    _d = pd.read_csv(os.path.join(RUN_DIR, f))
    assert _d.id.nunique() == n and {"p_a", "p_b", "p_c", "p_d"} <= set(_d.columns) and _d[["p_a", "p_b", "p_c", "p_d"]].notna().all().all(), f"{f}: 문항 {_d.id.nunique()}/{n} 또는 확률 열 이상"
_log = json.load(open(os.path.join(RUN_DIR, "train_log.json")))
assert _log["config"]["MODEL_ID"] == MODEL_ID, _log["config"]
_ad = sorted(glob.glob(os.path.join(RUN_DIR, "adapter_ep*", "adapter_config.json")))
assert _ad, "최종 어댑터(adapter_ep*)가 없습니다"
_lp = os.path.join(OUT_DIR, "results_log_s2.csv")
if os.path.exists(_lp): shutil.copy(_lp, os.path.join(RUN_DIR, "results_log_s2_runpod.csv"))
api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
assert api.repo_info(HF_REPO, repo_type="dataset").private, f"{HF_REPO}가 공개 저장소입니다 → HF 웹에서 Private로 바꾼 뒤 다시 실행하세요"
t0 = time.time(); _pre = f"vqa_runs/{RUN_NAME}"
print(f"올리는 중: {RUN_DIR} → {HF_REPO}/{_pre} (체크포인트 제외)")
api.upload_folder(repo_id=HF_REPO, repo_type="dataset", folder_path=RUN_DIR, path_in_repo=_pre,
                  ignore_patterns=["ckpt_last/*", "ckpt_last/**", "ckpt_last_tmp/*", "ckpt_last_tmp/**"], commit_message=f"RunPod {RUN_NAME}")
_remote = set(api.list_repo_files(HF_REPO, repo_type="dataset"))
_need = [f"{_pre}/{f}" for f in _val + _tst + ["train_log.json"]] + [f"{_pre}/" + os.path.relpath(a, RUN_DIR).replace(os.sep, "/") for a in _ad]
_miss = [f for f in _need if f not in _remote]
assert not _miss, f"올라가지 않은 파일: {_miss}"
print(f"✅ 올리기 완료 ({time.time() - t0:.0f}초) | 검증 {_val} · test {_tst} · 어댑터 {len(_ad)}개")
print(f"   검증 정확도(epoch 끝): {_log['epoch_acc']}")
print("다음: ① Colab 브리지 노트북 B 단계로 Drive에 받기 → ② Drive에 들어온 것을 확인한 뒤 RunPod에서 이 pod를 Terminate")
